# 03 — Threshold Tuning

Tune threshold để tối ưu F2 **không cần rerun LLM**.

**Mục tiêu:**
- Thử nhiều giá trị `safe_threshold`, `high_conf_threshold`, `max_articles`
- Tính local F2 với dev set
- Tìm config tốt nhất trước khi dùng 1 trong 5 lượt Private Phase

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
from pathlib import Path

from vpl.settings import DATA_DIR, RESULTS_STREAM_FILE
from vpl.cache import RetrievalCache
from vpl.search.hybrid import ScoredChunk
from vpl.answer.postprocess import PostConfig, PostProcessor
from vpl.evaluate import macro_f2
from vpl.submit import load_jsonl_stream

In [ ]:
questions = json.loads((DATA_DIR / 'R2AIStage1DATA.json').read_text(encoding='utf-8'))
cache = RetrievalCache()
existing = {r['id']: r for r in load_jsonl_stream(RESULTS_STREAM_FILE)}
print(f'Questions: {len(questions)}, Cached: {len(cache.completed_ids)}, Generated: {len(existing)}')

## 1. Phân tích score distribution từ cache

In [ ]:
# Lấy score distribution để chọn threshold hợp lý
all_top1_scores = []
sample_ids = list(cache.completed_ids)[:300]

for qid in sample_ids:
    try:
        chunks = cache.get(qid)
        if chunks:
            all_top1_scores.append(float(chunks[0].get('score', 0)))
    except Exception:
        pass

arr = np.array(all_top1_scores)
print('Top-1 score distribution:')
for p in [10, 25, 50, 75, 90]:
    print(f'  p{p}: {np.percentile(arr, p):.3f}')

print(f'\nSuggested thresholds:')
print(f'  safe_threshold:      ~{np.percentile(arr, 20):.2f}')
print(f'  high_conf_threshold: ~{np.percentile(arr, 55):.2f}')

## 2. Retune function

In [ ]:
def retune(safe_threshold, high_conf_threshold, min_articles, max_articles):
    cfg = PostConfig(
        safe_threshold=safe_threshold,
        high_conf_threshold=high_conf_threshold,
        min_articles=min_articles,
        max_articles=max_articles,
    )
    processor = PostProcessor(cfg)
    results = []
    for q in questions:
        qid = int(q['id'])
        if qid not in existing or qid not in cache.completed_ids:
            continue
        try:
            raw = cache.get(qid)
        except KeyError:
            continue
        chunks = [ScoredChunk(chunk=c, score=float(c.get('score', 0))) for c in raw]
        result = processor.build_result(qid, q['question'], existing[qid]['answer'], chunks)
        results.append(result)
    return results

print('retune() ready')

## 3. Grid search (cần dev_set.json)

In [ ]:
DEV_SET = DATA_DIR / 'dev_set.json'

if not DEV_SET.exists():
    print('Chưa có dev_set.json.')
    print('Tạo bằng cách annotate thủ công 30-50 câu:')
    print('[{"id": 1, "relevant_articles": ["04/2017/QH14|...|Điều 4"]}, ...]')
else:
    dev_refs = json.loads(DEV_SET.read_text(encoding='utf-8'))
    dev_ids = {int(r['id']) for r in dev_refs}
    print(f'Dev set: {len(dev_refs)} questions')

    configs = [
        (0.20, 0.40, 3,  8),
        (0.25, 0.45, 3,  8),
        (0.30, 0.50, 3,  8),   # default
        (0.20, 0.40, 3, 10),
        (0.15, 0.35, 2, 10),
        (0.25, 0.45, 3, 12),
    ]

    best_f2, best_cfg = 0, None
    for safe, high, min_a, max_a in configs:
        res = retune(safe, high, min_a, max_a)
        dev_preds = [r for r in res if r['id'] in dev_ids]
        if not dev_preds:
            continue
        m = macro_f2(dev_preds, dev_refs)
        marker = ' ← BEST' if m['macro_f2'] > best_f2 else ''
        print(f'safe={safe} high={high} min={min_a} max={max_a} → F2={m["macro_f2"]:.4f} P={m["macro_precision"]:.4f} R={m["macro_recall"]:.4f}{marker}')
        if m['macro_f2'] > best_f2:
            best_f2, best_cfg = m['macro_f2'], (safe, high, min_a, max_a)

    print(f'\n✅ Best: safe={best_cfg[0]} high={best_cfg[1]} min={best_cfg[2]} max={best_cfg[3]} → F2={best_f2:.4f}')

## 4. Export retuned results

In [ ]:
# Sau khi tìm được config tốt, export kết quả
# Thay các giá trị bên dưới bằng best config từ grid search
SAFE = 0.25
HIGH = 0.45
MIN_A = 3
MAX_A = 8

from vpl.settings import OUTPUT_DIR
results = retune(SAFE, HIGH, MIN_A, MAX_A)
out = OUTPUT_DIR / f'results_retune_safe{SAFE}_max{MAX_A}.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(sorted(results, key=lambda r: r['id']), ensure_ascii=False, indent=2), encoding='utf-8')
print(f'✅ Exported {len(results)} results → {out}')